**Important**: This notebook is RAM intensive!

In [ ]:
!pip install -q datasets

In [ ]:
import os
import warnings
from datetime import datetime, date, timedelta
from pathlib import Path
import gc
import glob

import numpy as np
import pandas as pd
from huggingface_hub import login, HfApi, hf_hub_download
import kagglehub
from typing import List, Tuple, Optional, Union, Callable, Dict

# Warnings Configuration
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    os.environ["KAGGLE_USERNAME"] = user_secrets.get_secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = user_secrets.get_secret("KAGGLE_KEY")
    hf_token = user_secrets.get_secret("HF_TOKEN")
    print("Kaggle secrets loaded.")
except ImportError:
    os.environ["KAGGLE_USERNAME"] = "KAGGLE_USERNAME"
    os.environ["KAGGLE_KEY"] = "KAGGLE_KEY"
    hf_token = "HF_token"
except Exception as e:
    os.environ["KAGGLE_USERNAME"] = "KAGGLE_USERNAME"
    os.environ["KAGGLE_KEY"] = "KAGGLE_KEY"
    os.environ["HF_TOKEN"] = "HF_token"

hf_api = None
try:
    login(token=hf_token)
    hf_api = HfApi()
    print("Hugging Face login successful and API instance created.")
except Exception as e:
    print(f"Hugging Face login failed: {e}")

In [ ]:
# Downloading the Datasets which we processed in the previues steps.
# 15 Min is only added cuz it has the fips info as a file.
path1 = kagglehub.dataset_download("mahdiseddigh/dynamic-rhythms-yearly-15min-parquet-colab")
path2 = kagglehub.dataset_download("mahdiseddigh/wather-data-2014-24-county-info-parquet-dataset")
path3 = kagglehub.dataset_download("mahdiseddigh/rhythms-data-aggregated-leftjoin-data-parquet")

In [ ]:
AGGREGATED_DATA_DIR = Path('/root/.cache/kagglehub/datasets/mahdiseddigh/rhythms-data-aggregated-leftjoin-data-parquet/versions/1')

# Paths to raw weather data files
WEATHER_DATA_DIR = Path('/root/.cache/kagglehub/datasets/mahdiseddigh/wather-data-2014-24-county-info-parquet-dataset/versions/1') # Example Kaggle path - ADJUST
WEATHER_HOURLY_PATH = WEATHER_DATA_DIR / 'wather_data_2014_24_county_info_hourly.parquet'
WEATHER_DAILY_PATH = WEATHER_DATA_DIR / 'wather_data_2014_24_county_info_daily.parquet'

# Path to FIPS mapping file
FIPS_MAP_PATH = "/root/.cache/kagglehub/datasets/mahdiseddigh/dynamic-rhythms-yearly-15min-parquet-colab/versions/3/final_combined_county_fips_map.csv"
FIPS_MAP_PATH = Path(FIPS_MAP_PATH)

OUTPUT_WEATHER_ENRICHED_DIR = Path('/content/dynamic_rhythms_full_aggregated_weather_parquet/') # Or '/kaggle/working/' here we use content because its on colab
OUTPUT_WEATHER_ENRICHED_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def load_fips_map(fips_map_path: Path) -> Optional[pd.DataFrame]:
    """
    Loads, cleans, and formats a FIPS-to-county/state map from a CSV file.

    Reads a CSV, identifies FIPS, state, and county columns (checking potential FIPS names),
    cleans the data by dropping missing or non-numeric FIPS, and formats the result
    as a DataFrame indexed by FIPS (int) with 'county' and 'state' columns (category dtype).

    Args:
        fips_map_path (Path): The path to the FIPS map CSV file.

    Returns:
        pd.DataFrame or None: A DataFrame with FIPS index and 'county'/'state' columns on success.
            Returns None if the file is not found, missing required columns, no valid data
            remains after cleaning, or an error occurs.
    """
    print(f"Loading FIPS map from: {fips_map_path}")
    if not fips_map_path.exists():
        print(f"Error: FIPS map file not found at {fips_map_path}")
        return None
    try:
        fips_map_df = pd.read_csv(fips_map_path)
        potential_fips_cols = ['fips_numeric', 'fips', 'fips_code']
        fips_col_numeric = None
        for col in potential_fips_cols:
            if col in fips_map_df.columns:
                fips_col_numeric = col
                break

        state_col = 'state'
        county_col = 'county'

        if fips_col_numeric is None or \
           state_col not in fips_map_df.columns or \
           county_col not in fips_map_df.columns:
            print(f"Error: FIPS map CSV missing required columns (one of {potential_fips_cols}, '{county_col}', '{state_col}')")
            print(f"  Columns found: {fips_map_df.columns.tolist()}")
            return None

        # Drop rows with missing essential data
        fips_map_df.dropna(subset=[fips_col_numeric, state_col, county_col], inplace=True)

        # Convert FIPS to numeric and drop rows where conversion fails
        fips_map_df[fips_col_numeric] = pd.to_numeric(fips_map_df[fips_col_numeric], errors='coerce')
        fips_map_df.dropna(subset=[fips_col_numeric], inplace=True)

        # Convert FIPS to integer
        if len(fips_map_df) == 0:
             print(f"Error: No valid FIPS entries remaining after cleaning.")
             return None
        fips_map_df[fips_col_numeric] = fips_map_df[fips_col_numeric].astype(np.int32)


        fips_map_df.set_index(fips_col_numeric, inplace=True)
        # Ensure we only keep county and state, handling potential different original names
        fips_map_df = fips_map_df[[county_col, state_col]].copy() # Using .copy() to avoid SettingWithCopyWarning

        fips_map_df.rename(columns={county_col: 'county', state_col: 'state'}, inplace=True)
        fips_map_df.index.name = 'fips'
        fips_map_df['state'] = fips_map_df['state'].astype('category')
        fips_map_df['county'] = fips_map_df['county'].astype('category')
        print(f"FIPS map loaded successfully ({len(fips_map_df)} entries). Index: {fips_map_df.index.name}, Cols: {fips_map_df.columns.tolist()}")
        return fips_map_df
    except Exception as e:
        print(f"Error loading or processing FIPS map: {e}")
        traceback.print_exc(limit=1)
        return None

In [ ]:
def load_aggregated_data(agg_data_path: Path, filename_for_index: str) -> Optional[pd.DataFrame]:
    """
    Loads, cleans, and formats an aggregated power/event dataset from a Parquet file.

    Reads a Parquet file, ensuring it has a 'fips_code', 'time' MultiIndex.
    If these columns exist but are not the index, it sets them as the index.
    It attempts to convert 'fips_code' to integer and 'time' to datetime,
    dropping rows where conversion fails or index values are missing.
    It also casts specific columns ('event_count_*', 'customers_out') to numeric
    and 'county'/'state' to category if they exist.

    Args:
        agg_data_path (Path): The path to the aggregated data Parquet file.
        filename_for_index (str): A descriptive string (like the filename) used
                                  in print statements for identification.

    Returns:
        pd.DataFrame or None: A pandas DataFrame with a ('fips_code', 'time')
            MultiIndex and cleaned data types on successful loading and processing.
            Returns None if the file is not found, the index structure is unexpected,
            no valid data remains after cleaning, or any other error occurs.
    """
    print(f"Loading aggregated data from: {agg_data_path}")
    if not agg_data_path.exists():
        print(f"Error: Aggregated data file not found at {agg_data_path}")
        return None
    try:
        df = pd.read_parquet(agg_data_path)
        expected_index_cols = ['fips_code', 'time']

        # Check if index columns are in DataFrame columns (need to be set as index)
        if all(col in df.columns for col in expected_index_cols):
             print("  Index columns found in DataFrame columns. Setting MultiIndex...")
             # Convert 'time' before setting index
             if not pd.api.types.is_datetime64_any_dtype(df['time']):
                  df['time'] = pd.to_datetime(df['time'], errors='coerce')
                  df.dropna(subset=['time'], inplace=True)
             # Convert 'fips_code' before setting index
             if not pd.api.types.is_integer_dtype(df['fips_code']):
                  df['fips_code'] = pd.to_numeric(df['fips_code'], errors='coerce').astype('Int32') # Use nullable Int32
                  df.dropna(subset=['fips_code'], inplace=True)
             # Drop original columns before setting index to avoid duplication if they exist as both
             df = df.drop(columns=[col for col in expected_index_cols if col in df.columns])
             df.set_index(expected_index_cols, inplace=True)

        # Check if index is already a MultiIndex with the expected names
        elif isinstance(df.index, pd.MultiIndex) and list(df.index.names) == expected_index_cols:
             print("  MultiIndex already set.")
             # Validate and convert index levels
             original_index = df.index
             try:
                 if not pd.api.types.is_integer_dtype(df.index.get_level_values('fips_code')):
                      print("  Warning: fips_code index level is not integer. Attempting conversion.")
                      # Convert to nullable Int32
                      new_fips_level = pd.to_numeric(df.index.get_level_values('fips_code'), errors='coerce').astype('Int32')
                      df.index = df.index.set_levels(new_fips_level, level='fips_code')

                 if not pd.api.types.is_datetime64_any_dtype(df.index.get_level_values('time')):
                      print("  Warning: time index level is not datetime. Attempting conversion.")
                      new_time_level = pd.to_datetime(df.index.get_level_values('time'), errors='coerce')
                      df.index = df.index.set_levels(new_time_level, level='time')

                 # Drop rows where conversion resulted in NaT or pd.NA in the index
                 df = df.loc[pd.notna(df.index.get_level_values('fips_code')) & pd.notna(df.index.get_level_values('time'))]

             except Exception as e:
                  print(f"Error during index level conversion/validation: {e}")
                  df.index = original_index
                  return None

        else:
            print(f"Error: Parquet file {agg_data_path.name} has unexpected index structure.")
            print(f"  Index: {df.index}")
            print(f"  Columns: {df.columns.tolist()}")
            return None

        # check if any data remains
        if len(df) == 0:
             print(f"Error: No valid entries remaining after setting/cleaning index for {filename_for_index}.")
             return None

        if not df.index.is_monotonic_increasing:
            print("  Sorting index...")
            df.sort_index(inplace=True)

        # Convert specific column types
        for col in df.columns:
            if col.startswith('event_count_'):
                 # Use nullable integer for potential NaNs
                 df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')
            elif col == 'customers_out':
                 df[col] = pd.to_numeric(df[col], errors='coerce', downcast='float')

        # Convert county/state to category if they exist
        if 'county' in df.columns:
             # Convert to object first to handle potential mixed types before categorizing
             df['county'] = df['county'].astype(str).astype('category')
        if 'state' in df.columns:
             # Convert to object first to handle potential mixed types before categorizing
             df['state'] = df['state'].astype(str).astype('category')


        print(f"Loaded '{filename_for_index}'. Shape: {df.shape}, Index: {df.index.names}")
        mem_usage = df.memory_usage(deep=True).sum() / (1024**2)
        print(f"  Memory usage: {mem_usage:.2f} MB")
        return df
    except Exception as e:
        print(f"Error loading aggregated data file {agg_data_path}: {e}")
        traceback.print_exc(limit=1)
        return None

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Optional
import traceback
import gc # Assuming gc is imported

def preprocess_weather_data(weather_path: Path,
                            resolution: str, # 'H' or 'D'
                            fips_map_df: Optional[pd.DataFrame]
                           ) -> pd.DataFrame:
    """
    Loads, preprocesses, and aggregates weather data from a Parquet file.

    Reads a Parquet file containing weather data. Cleans the data by dropping
    rows with missing essential columns ('fips', 'date', and 'hour' if resolution
    is 'H'). Converts 'fips' to integer. Creates a 'time' column by combining
    'date' and 'hour' (for 'H') or using normalized 'date' (for 'D'). Aggregates
    numeric weather columns ('tavg', 'tmin', 'tmax', 'prcp', 'snow', 'wspd')
    by taking the mean grouped by ('fips', 'time'). Optionally merges 'state'
    information from the provided FIPS map DataFrame based on 'fips'.
    The resulting DataFrame is indexed by ('fips', 'time').

    Args:
        weather_path (Path): The path to the weather data Parquet file.
        resolution (str): The desired time resolution for aggregation. Must be
                          'H' for hourly or 'D' for daily.
        fips_map_df (Optional[pd.DataFrame]): A DataFrame mapping FIPS codes
                                             to state information. Expected to
                                             be indexed by FIPS (int) and contain
                                             a 'state' column. If None or empty,
                                             the 'state' column is not added.

    Returns:
        pd.DataFrame: A pandas DataFrame with a ('fips', 'time') MultiIndex.
            Contains aggregated numeric weather columns (float32) and optionally
            a 'state' column (category). Returns an empty DataFrame if the file
            is not found, required columns are missing, or an error occurs
            during processing.

    Raises:
        ValueError: If the `resolution` argument is not 'H' or 'D'.
    """
    print(f"\n--- Preprocessing Weather Data ({resolution=}) from: {weather_path} ---")
    if not weather_path.exists():
        print(f"Warning: Weather file not found at {weather_path}. Returning empty DataFrame.")
        return pd.DataFrame()

    add_state = fips_map_df is not None and not fips_map_df.empty
    if not add_state:
         print("Warning: Valid FIPS map not provided. Cannot add 'state' column.")

    numeric_weather_cols = ['tavg', 'tmin', 'tmax', 'prcp', 'snow', 'wspd']
    base_cols_to_load = ['date', 'fips']
    hour_col_name = 'hour' # Name of the column containing the hour value
    if resolution == 'H':
         # For hourly, we need date, fips, hour, and the numeric columns
         weather_cols_to_load = base_cols_to_load + [hour_col_name] + numeric_weather_cols
    elif resolution == 'D':
         # For daily, we need date, fips, and the numeric columns
         weather_cols_to_load = base_cols_to_load + numeric_weather_cols
    else:
        raise ValueError(f"Invalid resolution '{resolution}'. Resolution must be 'H' or 'D'.")


    try:
        df_weather = pd.read_parquet(weather_path)
        print(f"Loaded raw weather data shape: {df_weather.shape}")

        actual_cols_present = df_weather.columns.tolist()
        # Only use columns that are both needed and present
        use_cols = [col for col in weather_cols_to_load if col in actual_cols_present]
        # Filter numeric_weather_cols to only include those actually present and used
        numeric_weather_cols = [col for col in numeric_weather_cols if col in use_cols]

        # Define columns absolutely required for processing based on resolution
        required_for_processing = ['date', 'fips'] + numeric_weather_cols
        if resolution == 'H':
            required_for_processing.append(hour_col_name)

        # Check if all required columns are in the columns we plan to use
        if not all(col in use_cols for col in required_for_processing):
             missing_req = [col for col in required_for_processing if col not in use_cols]
             print(f"Error: Missing required columns in weather data: {missing_req}. Found: {actual_cols_present}")
             return pd.DataFrame()

        print(f"Using weather columns: {use_cols}")
        df_weather = df_weather[use_cols].copy()
        gc.collect()

        # Drop rows missing essential index/time components
        base_dropna_cols = ['fips', 'date']
        if resolution == 'H':
            base_dropna_cols.append(hour_col_name)
        df_weather.dropna(subset=base_dropna_cols, inplace=True)

        # Clean and convert FIPS
        df_weather['fips'] = pd.to_numeric(df_weather['fips'], errors='coerce')
        df_weather.dropna(subset=['fips'], inplace=True)
        if len(df_weather) == 0:
             print("Warning: No valid FIPS entries remaining after cleaning.")
             return pd.DataFrame()
        df_weather['fips'] = df_weather['fips'].astype(np.int32)


        # Create time index based on resolution
        if resolution == 'H':
            print("  Combining date and hour columns for hourly timestamp...")
            df_weather['date_dt'] = pd.to_datetime(df_weather['date'], errors='coerce').dt.normalize()
            # Ensure hour_col_name exists before converting
            if hour_col_name not in df_weather.columns:
                 print(f"Error: Hourly resolution requires '{hour_col_name}' column, which is missing.")
                 return pd.DataFrame()
            df_weather['hour_num'] = pd.to_numeric(df_weather[hour_col_name], errors='coerce')
            df_weather.dropna(subset=['date_dt', 'hour_num'], inplace=True)
            if len(df_weather) == 0:
                 print("Warning: No valid date/hour entries remaining after cleaning.")
                 return pd.DataFrame()
            df_weather['hour_num'] = df_weather['hour_num'].astype(int)
            df_weather['time'] = df_weather['date_dt'] + pd.to_timedelta(df_weather['hour_num'], unit='h')
            # Drop original columns used to create 'time'
            df_weather.drop(columns=['date', hour_col_name, 'date_dt', 'hour_num'], inplace=True, errors='ignore') # errors='ignore' in case a column was already dropped
        elif resolution == 'D':
            print("  Using normalized date for daily timestamp...")
            df_weather['time'] = pd.to_datetime(df_weather['date'], errors='coerce').dt.normalize()
            df_weather.drop(columns=['date'], inplace=True, errors='ignore') # errors='ignore' in case date was not in use_cols
        # Drop rows where time conversion failed
        df_weather.dropna(subset=['time'], inplace=True)
        if len(df_weather) == 0:
             print(f"Warning: No valid time entries remaining after cleaning for resolution '{resolution}'.")
             return pd.DataFrame()

        # Ensure numeric weather columns are numeric before aggregation
        for col in numeric_weather_cols:
             if col in df_weather.columns: # Check if the column is actually present in the filtered df
                df_weather[col] = pd.to_numeric(df_weather[col], errors='coerce')
        # Drop rows where all numeric weather columns are NaN AFTER conversion attempts
        # This prevents dropping rows that only have FIPS/Time but no weather data,
        # but cleans up rows where weather data failed conversion.
        df_weather.dropna(subset=numeric_weather_cols, how='all', inplace=True)
        if len(df_weather) == 0:
             print("Warning: No entries with valid weather data remaining after cleaning.")
             return pd.DataFrame()


        print(f"Aggregating weather stations by mean over ('fips', 'time')...")
        # Group and aggregate only the numeric weather columns that were identified as present
        df_weather_agg = df_weather.groupby(['fips', 'time'], observed=True)[numeric_weather_cols].mean()
        df_weather_agg = df_weather_agg.astype(np.float32)

        del df_weather
        gc.collect()

        if add_state:
            print("Adding 'state' column from FIPS map...")
            # Ensure fips_map_df index is int32 for consistent merge
            if fips_map_df.index.dtype != np.int32:
                try:
                    fips_map_df.index = fips_map_df.index.astype(np.int32)
                except Exception as e:
                     print(f"Error converting FIPS map index to int32: {e}. Skipping state merge.")
                     add_state = False # Disable adding state
            if add_state: # Check add_state flag again
                # Perform the merge using left index (fips) and right index (fips)
                original_rows = len(df_weather_agg)
                df_weather_agg = df_weather_agg.merge(fips_map_df[['state']],
                                                    left_index=True, right_index=True, how='left')
                missing_states = df_weather_agg['state'].isna().sum()
                if missing_states > 0:
                    print(f"Warning: Could not find state for {missing_states} FIPS-time entries in the weather data.")
                # Convert state to category if it was successfully added
                if 'state' in df_weather_agg.columns:
                     df_weather_agg['state'] = df_weather_agg['state'].astype('category')
        else:
            print("Skipping adding 'state' column.")

        # Sort the index
        if not df_weather_agg.index.is_monotonic_increasing:
             print("Sorting weather index...")
             df_weather_agg.sort_index(inplace=True)

        mem_usage = df_weather_agg.memory_usage(deep=True).sum() / (1024**2)
        print(f"Weather preprocessing complete ({resolution=}). Shape: {df_weather_agg.shape}, Memory: {mem_usage:.2f} MB")
        gc.collect()
        return df_weather_agg

    except Exception as e:
        print(f"Error processing weather file {weather_path}: {e}")
        traceback.print_exc(limit=1)
        return pd.DataFrame()

In [ ]:
def preprocess_weather_data_weekly(daily_weather_df: pd.DataFrame,
                                   fips_map_df: Optional[pd.DataFrame]
                                   ) -> pd.DataFrame:
    """
    Aggregates preprocessed daily weather data to weekly summaries.

    Args:
        daily_weather_df: DataFrame containing preprocessed daily weather
                          with a MultiIndex ('fips', 'time').
        fips_map_df: DataFrame mapping FIPS to state (needed if adding state).

    Returns:
        DataFrame with weekly aggregated weather data.
    """
    print("\n--- Aggregating Daily Weather to Weekly Summaries ---")
    if daily_weather_df is None or daily_weather_df.empty:
        print("Warning: Input daily weather DataFrame is empty. Cannot create weekly aggregates.")
        return pd.DataFrame()

    # Check if required columns exist
    required_cols = ['tavg', 'tmin', 'tmax', 'prcp', 'snow', 'wspd']
    if not all(col in daily_weather_df.columns for col in required_cols):
        missing = [col for col in required_cols if col not in daily_weather_df.columns]
        print(f"Warning: Daily weather data missing required columns for weekly aggregation: {missing}. Skipping weekly processing.")
        return pd.DataFrame()

    # --- Define Weekly Aggregations ---
    # Calculate mean, min, max for temps; sum for precip/snow; mean for wind
    agg_dict = {
        'tavg': 'mean',
        'tmin': 'min',
        'tmax': 'max',
        'prcp': 'sum',
        'snow': 'sum',
        'wspd': 'mean'
    }
    # Generate new column names
    new_column_names = {
        ('tavg', 'mean'): 'tavg_weekly_mean',
        ('tmin', 'min'): 'tmin_weekly_min',
        ('tmax', 'max'): 'tmax_weekly_max',
        ('prcp', 'sum'): 'prcp_weekly_sum',
        ('snow', 'sum'): 'snow_weekly_sum',
        ('wspd', 'mean'): 'wspd_weekly_mean'
    }

    # --- Perform Aggregation ---
    try:
        print("Grouping daily weather by FIPS and week (starting Monday)...")
        # Make sure index is sorted before resampling/grouping
        if not daily_weather_df.index.is_monotonic_increasing:
            daily_weather_df = daily_weather_df.sort_index()

        # Reset index to use pd.Grouper
        daily_weather_reset = daily_weather_df.reset_index()

        week_start_freq = 'W-MON'
        print(f"Using weekly frequency: '{week_start_freq}' (Ensure this matches target data's weekly aggregation)")

        df_weekly_agg = daily_weather_reset.groupby(
            ['fips', pd.Grouper(key='time', freq=week_start_freq)]
        ).agg(agg_dict)

        # Rename the multi-level columns created by agg
        df_weekly_agg.columns = ["_".join(col).strip() for col in df_weekly_agg.columns.values] # Flatten like ('tmin', 'min')
        # Apply more descriptive names
        df_weekly_agg.rename(columns={ # Rename based on agg_dict keys
            'tavg_mean': 'tavg_weekly_mean',
            'tmin_min': 'tmin_weekly_min',
            'tmax_max': 'tmax_weekly_max',
            'prcp_sum': 'prcp_weekly_sum',
            'snow_sum': 'snow_weekly_sum',
            'wspd_mean': 'wspd_weekly_mean'
            }, inplace=True)

        # Convert to float32 AFTER aggregation
        df_weekly_agg = df_weekly_agg.astype(np.float32)

        del daily_weather_reset # Free memory
        gc.collect()

        # Add state column if FIPS map is available
        add_state = fips_map_df is not None and not fips_map_df.empty
        if add_state:
            print("Adding 'state' column from FIPS map to weekly weather...")
            if fips_map_df.index.dtype != np.int32:
                fips_map_df.index = fips_map_df.index.astype(np.int32)
            # Merge on the 'fips' level of the MultiIndex
            df_weekly_agg = df_weekly_agg.merge(fips_map_df[['state']],
                                                left_on='fips', # Index level 0
                                                right_index=True, how='left')
            missing_states = df_weekly_agg['state'].isna().sum()
            if missing_states > 0:
                 print(f"Warning: Could not find state for {missing_states} FIPS-week entries.")
            if 'state' in df_weekly_agg.columns:
                 df_weekly_agg['state'] = df_weekly_agg['state'].astype('category')
        else:
            print("Skipping adding 'state' column to weekly weather.")

        # Ensure index is sorted
        if not df_weekly_agg.index.is_monotonic_increasing:
             print("Sorting weekly weather index...")
             df_weekly_agg.sort_index(inplace=True)

        mem_usage = df_weekly_agg.memory_usage(deep=True).sum() / (1024**2)
        print(f"Weekly weather aggregation complete. Shape: {df_weekly_agg.shape}, Memory: {mem_usage:.2f} MB")
        gc.collect()
        return df_weekly_agg

    except Exception as e:
        print(f"Error aggregating daily weather to weekly: {e}")
        traceback.print_exc(limit=1)
        return pd.DataFrame()

In [ ]:
def add_weather_features(target_df: pd.DataFrame,
                         weather_df_agg: pd.DataFrame,
                         fips_map_df: pd.DataFrame,
                         resolution_name: str
                         ) -> pd.DataFrame:
    """
    Merges aggregated weather data into a target DataFrame and approximates missing entries.

    Takes a target DataFrame (expected to be indexed by 'fips_code', 'time') and
    performs a left merge with aggregated weather data (expected to be indexed
    by 'fips', 'time'). For rows in the target DataFrame that do not find a direct
    match in the weather data, it attempts to approximate the missing weather
    values using the mean weather for the corresponding state at that time.
    This approximation requires a 'state' column to be present in the aggregated
    weather DataFrame. If the target DataFrame lacks a 'state' column needed for
    the approximation step, it tries to merge it from the provided fips_map_df.
    Any weather values still missing after the state-level approximation are filled
    with 0. A 'weather_source' column is added to indicate whether the weather
    data for each row came from a 'direct_match', 'approx_state_avg', 'approx_fill_0',
    or 'unknown' source.

    Args:
        target_df (pd.DataFrame): The DataFrame to which weather features will be added.
                                  Must have a ('fips_code', 'time') MultiIndex with
                                  an integer 'fips_code' level.
        weather_df_agg (pd.DataFrame): Aggregated weather data, typically from
                                      `preprocess_weather_data`. Must have a
                                      ('fips', 'time') MultiIndex with an integer
                                      'fips' level and ideally includes a 'state'
                                      column for state-level approximation.
        fips_map_df (pd.DataFrame): A DataFrame mapping FIPS codes to state
                                    information. Expected to be indexed by FIPS
                                    (matching target_df fips_code dtype) and
                                    contain a 'state' column. Used if target_df
                                    needs the 'state' column added for approximation.
        resolution_name (str): A descriptive string (e.g., 'hourly', 'daily') used
                               in print statements for identification during processing.

    Returns:
        pd.DataFrame: The `target_df` DataFrame with added weather columns. These
            columns will contain merged data where available, approximated values
            using state averages, or 0 if approximation was not possible. Includes
            a 'weather_source' column. Returns the original `target_df` if inputs
            are empty or index structure/dtype mismatches are detected at the start.
    """
    print(f"\n--- Adding Weather Features to {resolution_name} dataset ---")
    start_time = time.time()
    if weather_df_agg.empty or target_df.empty:
        print(f"[{resolution_name}] Input DataFrame(s) empty. Skipping weather addition.")
        return target_df

    target_fips_level_name = 'fips_code'
    weather_fips_level_name = 'fips' # Expected FIPS level name in weather_df_agg
    time_level_name = 'time'

    # --- Validate Input DataFrames ---
    if not (isinstance(target_df.index, pd.MultiIndex) and list(target_df.index.names) == [target_fips_level_name, time_level_name]):
        print(f"[{resolution_name}] Error: Target df index {target_df.index.names} != ['{target_fips_level_name}', '{time_level_name}']")
        return target_df
    if not pd.api.types.is_integer_dtype(target_df.index.get_level_values(target_fips_level_name)):
        print(f"[{resolution_name}] Error: Target fips_code index level is not integer.")
        return target_df
    # Optional: Could also check if time level is datetime

    # Ensure weather_df_agg has the expected index, possibly after resetting 'state'
    current_weather_index_names = list(weather_df_agg.index.names)
    if not (isinstance(weather_df_agg.index, pd.MultiIndex) and
            # Check if index is ('fips', 'time') exactly OR ('state', 'fips', 'time') allowing state reset
            (current_weather_index_names == [weather_fips_level_name, time_level_name] or
             current_weather_index_names == ['state', weather_fips_level_name, time_level_name] )
           ):
         print(f"[{resolution_name}] Error: Weather df index {current_weather_index_names} unexpected structure. Expected ['{weather_fips_level_name}', '{time_level_name}'] or ['state', '{weather_fips_level_name}', '{time_level_name}']")
         return target_df

    # If weather_df_agg index includes 'state', reset it to make index ('fips', 'time')
    if 'state' in weather_df_agg.index.names:
         weather_df_agg = weather_df_agg.reset_index(level='state')
         print(f"[{resolution_name}] Reset 'state' level from weather index. New index: {list(weather_df_agg.index.names)}")


    if not pd.api.types.is_integer_dtype(weather_df_agg.index.get_level_values(weather_fips_level_name)):
        print(f"[{resolution_name}] Error: Weather fips index level is not integer.")
        return target_df
    # Optional: Could also check weather time level

    # --- Identify weather columns to merge ---
    # Exclude 'state' column from merge if it exists in weather_df_agg (it will be used for approximation later)
    weather_cols_to_add = [col for col in weather_df_agg.columns if col != 'state']
    print(f"[{resolution_name}] Weather columns to merge: {weather_cols_to_add}")
    if not weather_cols_to_add:
        print(f"[{resolution_name}] Warning: No weather columns identified for merge. Skipping weather addition.")
        return target_df

    # --- Initial Merge ---
    print(f"[{resolution_name}] Performing initial left merge (Target LEFT JOIN Weather)...")
    # Rename target FIPS level to match weather FIPS level temporarily for merge
    target_df_renamed = target_df.rename_axis(index={target_fips_level_name: weather_fips_level_name}).copy()

    # Select only weather columns (excluding state) for the merge
    weather_cols_for_merge = weather_df_agg[weather_cols_to_add]

    df_merged = target_df_renamed.merge(
        weather_cols_for_merge,
        left_index=True,
        right_index=True,
        how='left'
    )
    # Rename FIPS index level back to original name
    df_merged = df_merged.rename_axis(index={weather_fips_level_name: target_fips_level_name})

    del target_df_renamed, weather_cols_for_merge
    gc.collect()
    print(f"[{resolution_name}] Shape after initial merge: {df_merged.shape}")

    # --- Handle Missing Weather Data ---
    # Check for NaNs in the *first* weather column as an indicator of missing weather for the row
    first_weather_col = weather_cols_to_add[0]
    if first_weather_col not in df_merged.columns:
        print(f"[{resolution_name}] Error: First weather column '{first_weather_col}' not found in merged df.")
        # Decide how to handle - perhaps fill all with 0 and mark source 'unknown'
        df_merged['weather_source'] = 'unknown' # Cannot proceed with filling
        return df_merged

    missing_weather_mask = df_merged[first_weather_col].isna()
    num_missing = missing_weather_mask.sum()

    if num_missing == 0:
        print(f"[{resolution_name}] No missing weather data found after merge.")
        # Add weather_source column for consistency, marked as direct match
        df_merged['weather_source'] = 'direct_match'
    else:
        print(f"[{resolution_name}] Found {num_missing} rows missing weather data after merge. Approximating using state averages...")

        # Ensure target_df has 'state' column for state aggregation
        if 'state' not in df_merged.columns:
            print(f"[{resolution_name}] 'state' column missing in target DataFrame, attempting merge from fips_map...")
            if fips_map_df is not None and not fips_map_df.empty and 'state' in fips_map_df.columns:
                 # Ensure fips_map_df index dtype matches target_df index level 0 dtype for merge
                 target_fips_dtype = df_merged.index.get_level_values(target_fips_level_name).dtype
                 if fips_map_df.index.dtype != target_fips_dtype:
                      try:
                          fips_map_df.index = fips_map_df.index.astype(target_fips_dtype)
                          print(f"[{resolution_name}] Converted fips_map_df index to {target_fips_dtype}.")
                      except Exception as e:
                           print(f"[{resolution_name}] Error converting fips_map_df index to {target_fips_dtype}: {e}. Cannot add 'state'.")
                           # Cannot proceed with state-level approximation
                           print(f"[{resolution_name}] Cannot perform state approximation due to missing/problematic 'state' info. Filling remaining NaNs with 0.")
                           df_merged.fillna({col: 0 for col in weather_cols_to_add}, inplace=True)
                           df_merged['weather_source'] = np.where(missing_weather_mask, 'approx_fill_0', 'direct_match')
                           if 'state' not in df_merged.columns: df_merged['state'] = pd.NA # Add a placeholder state col if it wasn't there
                           if 'state' in df_merged.columns: df_merged['state'] = df_merged['state'].astype('category')
                           return df_merged # Exit early after filling with 0

                 # Perform the merge to add the state column based on the fips_code index
                 original_merged_shape = df_merged.shape
                 df_merged = df_merged.merge(fips_map_df[['state']],
                                             left_on=target_fips_level_name, # Merge using the fips_code index level of df_merged
                                             right_index=True, # Merge with the index of fips_map_df
                                             how='left',
                                             validate='many_one') # A FIPS code can match only one state in the map
                 print(f"[{resolution_name}] Added 'state' from fips_map. Shape change: {original_merged_shape} -> {df_merged.shape}")
                 if 'state' in df_merged.columns:
                       missing_states_in_target = df_merged['state'].isna().sum()
                       if missing_states_in_target > 0:
                            print(f"[{resolution_name}] Warning: {missing_states_in_target} rows in target DataFrame could not be matched to a state from fips_map.")

            else:
                 print(f"[{resolution_name}] Error: 'state' column missing in target and no valid fips_map provided (or fips_map is empty/missing 'state'). Cannot perform state approximation.")
                 # Cannot proceed with state-level approximation
                 print(f"[{resolution_name}] Cannot perform state approximation due to missing 'state' info. Filling remaining NaNs with 0.")
                 df_merged.fillna({col: 0 for col in weather_cols_to_add}, inplace=True)
                 df_merged['weather_source'] = np.where(missing_weather_mask, 'approx_fill_0', 'direct_match')
                 if 'state' not in df_merged.columns: df_merged['state'] = pd.NA # Add a placeholder state col if it wasn't there
                 if 'state' in df_merged.columns: df_merged['state'] = df_merged['state'].astype('category')
                 return df_merged # Exit early after filling with 0


        # Check if state column is now available in the merged DataFrame
        if 'state' not in df_merged.columns:
             print(f"[{resolution_name}] Error: 'state' column is required for state-level approximation but is not available after attempting to add it.")
             # Fallback to filling with 0
             print(f"[{resolution_name}] Cannot perform state approximation. Filling remaining NaNs with 0.")
             df_merged.fillna({col: 0 for col in weather_cols_to_add}, inplace=True)
             df_merged['weather_source'] = np.where(missing_weather_mask, 'approx_fill_0', 'direct_match')

        elif 'state' not in weather_df_agg.columns:
             print(f"[{resolution_name}] Warning: 'state' column not found in aggregated weather data ({weather_df_agg.columns.tolist()}). Cannot calculate state averages for approximation.")
             # Cannot perform state-level approximation - Fallback to filling with 0
             print(f"[{resolution_name}] Cannot perform state approximation. Filling remaining NaNs with 0.")
             df_merged.fillna({col: 0 for col in weather_cols_to_add}, inplace=True)
             df_merged['weather_source'] = np.where(missing_weather_mask, 'approx_fill_0', 'direct_match')

        else:
            # --- Perform State-Time Approximation ---
            print(f"[{resolution_name}] Calculating state-time averages from weather data...")
            # Group weather data by state and time to calculate state-level averages
            # Ensure 'state' is a column in weather_df_agg for this groupby
            state_time_avg_weather = weather_df_agg.groupby(['state', time_level_name], observed=True)[weather_cols_to_add].mean()
            state_time_avg_weather = state_time_avg_weather.astype(np.float32)
            # Rename columns to avoid clash during merge back
            state_time_avg_weather.columns = [f"{col}_state_avg" for col in state_time_avg_weather.columns]
            print(f"[{resolution_name}] Calculated state-time averages. Shape: {state_time_avg_weather.shape}")

            print(f"[{resolution_name}] Mapping state averages to missing rows in merged df...")
            # Extract the rows from df_merged that are missing weather data and have a state/time index
            # Need to reset index to merge on 'state' and 'time' columns
            df_to_fill = df_merged.loc[missing_weather_mask].reset_index()

            # Merge the calculated state averages into the subset of rows needing fill
            df_filled_values = df_to_fill.merge(
                state_time_avg_weather,
                on=['state', time_level_name],
                how='left' # Use left merge to keep all rows from df_to_fill
            )
            # Set index back to ('fips_code', 'time') to align with df_merged
            df_filled_values.set_index([target_fips_level_name, time_level_name], inplace=True)

            print(f"[{resolution_name}] Filling NaNs in df_merged using mapped state averages...")
            # Create a dictionary mapping original column names to the Series containing state averages
            update_dict = {col: df_filled_values[f"{col}_state_avg"] for col in weather_cols_to_add}

            # Use update_dict with fillna. The fillna method will only fill NaN values
            df_merged.fillna(update_dict, inplace=True)

            # Mark the rows that were successfully filled by state average
            # A row is marked 'approx_state_avg' if it was initially missing AND is no longer NaN in the first weather column
            # (meaning it was filled by the state average or fallback 0)
            df_merged['weather_source'] = np.where(missing_weather_mask & ~df_merged[first_weather_col].isna(), 'approx_state_avg', 'direct_match')


            # --- Handle Remaining NaNs after State Approximation ---
            # Identify rows that were missing initially but *still* have NaNs after state approximation
            remaining_nan_mask = df_merged[first_weather_col].isna() # Check NaNs *after* state fillna
            num_remaining_nan = remaining_nan_mask.sum()

            if num_remaining_nan > 0:
                 print(f"[{resolution_name}] Warning: {num_remaining_nan} rows still have NaN weather data after state approximation. Filling with 0.")
                 fill_zero_cols = {col: 0 for col in weather_cols_to_add}
                 # Only fill NaNs in rows that were originally missing AND are still NaN
                 condition_fill_zero = missing_weather_mask & remaining_nan_mask
                 df_merged.loc[condition_fill_zero, weather_cols_to_add] = df_merged.loc[condition_fill_zero, weather_cols_to_add].fillna(0)

                 # Update weather_source for rows filled with 0
                 df_merged.loc[condition_fill_zero, 'weather_source'] = 'approx_fill_0'

            del state_time_avg_weather, df_to_fill, df_filled_values, update_dict
            gc.collect()

    # --- Final Type Conversions ---
    for col in weather_cols_to_add:
         # Ensure weather columns are float32, handling potential NaNs introduced by merge/fill
         if col in df_merged.columns:
             df_merged[col] = df_merged[col].astype(np.float32)

    # Convert weather_source and state to category if they exist
    if 'weather_source' in df_merged.columns:
        df_merged['weather_source'] = df_merged['weather_source'].astype('category')
    if 'state' in df_merged.columns:
        # Convert to object first to handle potential mixed types/NaT before categorizing
        df_merged['state'] = df_merged['state'].astype(str).astype('category')


    end_time = time.time()
    print(f"[{resolution_name}] Weather feature addition complete in {end_time - start_time:.2f} sec. Final shape: {df_merged.shape}")
    mem_usage = df_merged.memory_usage(deep=True).sum() / (1024**2)
    print(f"  Final memory usage: {mem_usage:.2f} MB")
    return df_merged

In [ ]:
from typing import Dict, Any, Tuple # Assuming these are imported
import pandas as pd # Assuming pandas is imported
from pathlib import Path # Assuming Path is imported
# Assuming load_aggregated_data, add_weather_features, traceback, gc, time are available in the context

def process_aggregation_level(
    name: str,
    config: Dict[str, Any],
    preprocessed_weather_dfs: Dict[str, pd.DataFrame],
    fips_map_df: pd.DataFrame,
    output_dir: Path
    ) -> Tuple[str, bool]:
    """
    Processes a single aggregation level configuration.

    Loads aggregated county-level data based on the configuration, optionally
    merges preprocessed weather features based on a specified weather key, and
    saves the resulting enriched DataFrame to a Parquet file. Handles loading
    and processing errors internally, returning a success status.

    Args:
        name (str): The descriptive name of this aggregation level (e.g., 'hourly', 'daily').
                    Used in print statements.
        config (Dict[str, Any]): A dictionary containing configuration for this level,
                                 expected to include 'input_file', 'output_suffix',
                                 and optionally 'weather_key'.
        preprocessed_weather_dfs (Dict[str, pd.DataFrame]): A dictionary where keys are
                                                         weather keys (like 'hourly',
                                                         'daily', 'weekly') and values
                                                         are the preprocessed weather DataFrames
                                                         (or None/empty if failed).
        fips_map_df (pd.DataFrame): The DataFrame mapping FIPS codes to states. Used
                                    by `add_weather_features` for state-level approximation
                                    if needed.
        output_dir (Path): The directory where the output Parquet file should be saved.

    Returns:
        Tuple[str, bool]: A tuple containing the `name` of the aggregation level
                          and a boolean indicating if the processing (loading,
                          weather addition, and saving) was successful (True)
                          or encountered an error (False).
    """
    print(f"\n>>> Starting processing for: {name} <<<")
    success = False
    try:
        input_path = config['input_file']
        weather_key = config.get('weather_key') # 'hourly', 'daily', 'weekly' or None
        output_filename = f"combined_county_{config['output_suffix']}.parquet"
        output_path = output_dir / output_filename

        base_df = load_aggregated_data(input_path, name)
        if base_df is None or base_df.empty:
            print(f"[{name}] Skipping: Loading failed or empty dataframe.")
            return name, success

        weather_data_to_add = None
        if weather_key and weather_key in preprocessed_weather_dfs:
            weather_data_to_add = preprocessed_weather_dfs[weather_key]
            if weather_data_to_add is None or weather_data_to_add.empty:
                 print(f"[{name}] Warning: Preprocessed weather data for key '{weather_key}' is empty.")
                 weather_data_to_add = None # Treat as if no weather key was provided

        if weather_data_to_add is not None:
            # add_weather_features handles internal errors and returns df
            enriched_df = add_weather_features(base_df, weather_data_to_add, fips_map_df, name)
            # Check if weather addition itself resulted in an empty df (shouldn't, but robust)
            if enriched_df is None or enriched_df.empty:
                print(f"[{name}] Warning: Weather feature addition resulted in empty dataframe.")
                # Could potentially fall back to base_df here if desired, but current logic returns empty df
                # For now, stick to existing behavior: if it's empty after weather, don't save
                # Returning success=False is appropriate
        else:
            print(f"[{name}] No weather data specified or available. Proceeding without weather enrichment.")
            enriched_df = base_df.copy() # Use .copy() to avoid potential SettingWithCopyWarning
            if 'weather_source' not in enriched_df.columns:
                enriched_df['weather_source'] = 'no_weather_added'
                enriched_df['weather_source'] = enriched_df['weather_source'].astype('category')


        if not enriched_df.empty:
            print(f"[{name}] Saving final data to {output_path}...")
            try:
                # Ensure output directory exists
                output_dir.mkdir(parents=True, exist_ok=True)
                enriched_df.to_parquet(output_path, index=True, compression='snappy')
                print(f"[{name}] ... Saved {output_path}")
                success = True
            except Exception as save_err:
                print(f"[{name}] Error saving file {output_path}: {save_err}")
                traceback.print_exc(limit=1)
        else:
            print(f"[{name}] Skipping save: Final dataframe is empty.")

        # Explicitly delete large objects to free memory
        del base_df
        if 'enriched_df' in locals() and enriched_df is not None:
             del enriched_df
        if 'weather_data_to_add' in locals() and weather_data_to_add is not None:
             del weather_data_to_add

        gc.collect()
        print(f">>> Finished processing for: {name} | Success: {success} <<<")

    except Exception as worker_err:
         # This catches errors *outside* of load_aggregated_data/add_weather_features if they happen here
         print(f"!!! UNHANDLED EXCEPTION IN WORKER for {name}: {worker_err}")
         traceback.print_exc(limit=2)
         success = False # Ensure success is False on unexpected error

    return name, success

In [ ]:
if __name__ == "__main__":
    main_start_time = time.time()

    print("--- Starting Weather Enrichment Process (Parallel) ---")
    # Multiprocessing Configuration
    num_cores = os.cpu_count() or 1
    max_workers = min(max(1, num_cores // 2), 8) # max 8 is a good choice since this uses up to 180GB RAM!
    print(f"System CPU count: {num_cores}. Using max_workers: {max_workers}")

    # Load FIPS map(Important)
    fips_map = load_fips_map(FIPS_MAP_PATH)
    if fips_map is None:
        print("CRITICAL ERROR: Could not load FIPS map. Exiting.")
        sys.exit(1)

    # Preprocess weather data (hourly, daily, THEN weekly)
    preprocessed_weather = {}
    preprocessed_weather['hourly'] = preprocess_weather_data(WEATHER_HOURLY_PATH, 'H', fips_map)
    preprocessed_weather['daily'] = preprocess_weather_data(WEATHER_DAILY_PATH, 'D', fips_map)
    preprocessed_weather['weekly'] = preprocess_weather_data_weekly(
        preprocessed_weather['daily'],
        fips_map
    )

    files_to_enrich_config = {
        "hourly_mean": {'input_file': AGGREGATED_DATA_DIR / 'dynamic_rhythms_data_aggregated_leftjoin_hourly_mean.parquet', 'weather_key': 'hourly', 'output_suffix': 'hourly_mean_weather'},
        "hourly_sum":  {'input_file': AGGREGATED_DATA_DIR / 'dynamic_rhythms_data_aggregated_leftjoin_hourly_sum.parquet',  'weather_key': 'hourly', 'output_suffix': 'hourly_sum_weather'},
        "daily_mean":  {'input_file': AGGREGATED_DATA_DIR / 'dynamic_rhythms_data_aggregated_leftjoin_daily_mean.parquet',  'weather_key': 'daily',  'output_suffix': 'daily_mean_weather'},
        "daily_sum":   {'input_file': AGGREGATED_DATA_DIR / 'dynamic_rhythms_data_aggregated_leftjoin_daily_sum.parquet',   'weather_key': 'daily',  'output_suffix': 'daily_sum_weather'},
        "weekly_mean": {'input_file': AGGREGATED_DATA_DIR / 'dynamic_rhythms_data_aggregated_leftjoin_weekly_mean.parquet', 'weather_key': 'weekly',  'output_suffix': 'weekly_mean_weather'},
        "weekly_sum":  {'input_file': AGGREGATED_DATA_DIR / 'dynamic_rhythms_data_aggregated_leftjoin_weekly_sum.parquet',  'weather_key': 'weekly',  'output_suffix': 'weekly_sum_weather'},
        }

    # Parallel Execution(Saves time)
    futures = []
    results_summary = {}

    executor = concurrent.futures.ProcessPoolExecutor(max_workers=max_workers)
    try:
        print(f"\nSubmitting {len(files_to_enrich_config)} tasks to {max_workers} workers...")
        for name, config in files_to_enrich_config.items():
            if not config['input_file'].exists():
                 print(f"Warning: Input file not found for '{name}': {config['input_file']}. Skipping task submission.")
                 results_summary[name] = False
                 continue
            # Check if required weather data exists before submitting
            weather_key = config.get('weather_key')
            if weather_key and (weather_key not in preprocessed_weather or preprocessed_weather[weather_key].empty):
                print(f"Warning: Required preprocessed weather data '{weather_key}' not available or empty for task '{name}'. Skipping submission.")
                results_summary[name] = False
                continue

            future = executor.submit(
                process_aggregation_level,
                name=name,
                config=config,
                preprocessed_weather_dfs=preprocessed_weather,
                fips_map_df=fips_map,
                output_dir=OUTPUT_WEATHER_ENRICHED_DIR
            )
            futures.append(future)

        print("All tasks submitted. Waiting for results...")
        for future in concurrent.futures.as_completed(futures):
            try:
                name_res, success_res = future.result()
                results_summary[name_res] = success_res
            except Exception as exc:
                print(f'\n!!! Error retrieving result from a worker: {exc}')

    finally:
        print("Shutting down worker processes...")
        executor.shutdown(wait=True)
        print("Workers shut down.")

    # Final Summary
    print("\n--- Processing Summary ---")
    successful_tasks = [name for name, success in results_summary.items() if success]
    failed_tasks = [name for name, success in results_summary.items() if not success]
    print(f"Successfully processed: {successful_tasks}")
    if failed_tasks:
        print(f"Failed or skipped: {failed_tasks}")

    # Final Cleanup
    print("\nCleaning up preprocessed weather data...")
    del preprocessed_weather, fips_map
    gc.collect()
    print(f"\n--- Uploading results directory to Kaggle Datasets ---")

    kaggle_username = os.environ.get("KAGGLE_USERNAME")
    if not kaggle_username:
        print("Warning: KAGGLE_USERNAME environment variable not set. Using placeholder.")
        kaggle_username = "your_kaggle_username"

    dataset_name = "dynamic-rhythms-weather-added"
    kaggle_handle = f"{kaggle_username}/{dataset_name}"
    KAGGLE_AVAILABLE = True
    output_dir = OUTPUT_WEATHER_ENRICHED_DIR
    if KAGGLE_AVAILABLE and kaggle_username != "your_kaggle_username":
        try:
            print(f"Attempting Kaggle dataset upload/update for handle: {kaggle_handle}")
            print(f"Uploading content from: {output_dir}")
            # Check what files are actually in the directory before upload
            local_files_for_kaggle = list(output_dir.glob('*'))
            if not local_files_for_kaggle:
                 print("Warning: Output directory is empty. Nothing to upload to Kaggle.")
            else:
                print(f"Files found in output directory for Kaggle upload: {[f.name for f in local_files_for_kaggle]}")
                kagglehub.dataset_upload(kaggle_handle, str(output_dir))
                print(f"Kaggle dataset upload/update initiated for handle: {kaggle_handle}")
                print("Note: Check Kaggle UI (kaggle.com/datasets/{kaggle_handle}) for upload progress and status.")
        except Exception as kaggle_err:
            print(f"Error uploading to Kaggle Datasets: {kaggle_err}")
            print("Ensure KAGGLE_USERNAME and KAGGLE_KEY environment variables are correctly set,")
            print("and the kagglehub library is installed and authenticated.")
            traceback.print_exc(limit=2)

    main_end_time = time.time()
    print("\n" + "="*60)
    print(f"--- Weather Enrichment Script Finished in {(main_end_time - main_start_time) / 60:.2f} minutes ---")
    print("="*60)

In [ ]:
# Colab Link: https://colab.research.google.com/drive/1nKbBj5NHa5rsuPv39t4GTOpVAps9yQKz?usp=sharing